# 08. Laya 체험 — 다국어 System 1 decision engine

[Laya](https://github.com/NandhaKishorM/laya)는 Convai Innovations가 공개한 오픈소스(Apache-2.0) decision model 패밀리입니다.

- **non-autoregressive**: BERT 계열 encoder(English 421M / multilingual 322M)가 한 번의 forward pass로 choice/noul/score에 답합니다 — 지금까지 본 decision model 중 가장 작습니다 (CPU로도 실용적).
- **다국어 100+**: `Router()`가 입력 언어를 감지해 한국어 등 비영어 입력을 `laya-multilingual` checkpoint로 자동 라우팅합니다.
- RLCD(strictly proper scoring rule 기반 RL)로 학습된 **calibrated 확률** 반환.
- `laya[serve]`는 **`/v1/systemone`** (Jev 호환 wire format)도 제공하고, LangChain/LangGraph/LlamaIndex/CrewAI/MCP 통합, TypeScript 포트(`npm install laya-ts`)까지 있습니다.
- 완전 무료 self-host — API key/계정 불필요.

이 노트북은 HTTP 서버 없이 **Python native `Router` API를 kernel 안에서 직접** 사용합니다 (kernel 환경에 `pip install laya` 필요 — workshop kernel에는 사전 설치되어 있으며, checkpoint는 HF_HOME 캐시에 다운로드됩니다).

In [ ]:
import sys, os, time
from pathlib import Path; sys.path.insert(0, str(Path.cwd().parent))  # ws_common 경로
os.environ.setdefault("HF_HOME", os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface")))
import ws_common as W
from laya import Router

router = Router()    # 첫 사용 시 checkpoint 다운로드
print("laya Router ready")

## 1. 한국어 질문 하나로 판단 3종을 한 번에

Laya는 한 state에 여러 질문을 묶어 보낼 수 있습니다. 금융 상담 문맥으로 **업무 분류(choice) + 긴급성(noul) + 고객 감정(score)** 을 동시에 물어봅니다. `routing.model`이 `multilingual`로 찍히는지(한국어 자동 감지) 확인하세요.

In [ ]:
state = "지갑을 도둑맞았어요! 카드 정지시켜 주세요. 빨리요!!"
questions = {
    "d2_team": {"type": "choice",
                "instructions": "Which banking support specialist team should handle this customer message?",
                "criteria": {c: None for c in
                             ["cards", "card_security", "disputes_fees", "transfers",
                              "accounts_balance", "top_up", "exchange_fx", "refunds",
                              "declined_pay", "identity_verif", "account_mgmt", "general_faq"]}},
    "urgent": {"type": "noul", "instructions": "Does this message convey urgency?"},
    "emotion": {"type": "score", "instructions": "How upset is the customer?",
                "criteria": ["calm", "worried", "very upset"]},
}

router.predict(state, questions)   # warm-up (측정 제외)
t0 = time.perf_counter()
r = router.predict(state, questions)
ms = (time.perf_counter() - t0) * 1000

print(f"checkpoint: {r['routing']['model']}   (질문 3개 합산 {ms:,.1f}ms)")
a = r["answers"]
print(f"  d2_team -> {a['d2_team']['choice']} (confidence {a['d2_team']['confidence']:.3f})")
print(f"  urgent  -> P(yes) = {a['urgent']['noul']:.3f}")
print(f"  emotion -> score {a['emotion']['score']:.2f}  (0=calm ~ 2=very upset)")

## 2. 우리 LangGraph workflow에 꽂기 — adapter 20줄

Laya는 Python 객체 API이므로, `.decide()` 서명에 맞춘 **얇은 adapter**만 쓰면 지금까지의 graph에 그대로 들어갑니다. backend 교체가 얼마나 작은 일인지 보여주는 예입니다.
> adapter에서 `model="multilingual"`을 **고정**합니다. 자동 라우팅에 맡기면 영어 섞인 긴 state(tool JSON 등)가 english checkpoint로 가면서 체크포인트 전환이 반복돼 느려지고, 긴 입력은 `max_len=8192`가 필요합니다. 또한 Laya는 **선택지 11개 이상 choice의 confidence는 uncalibrated**라고 스스로 경고합니다(D2 12지 해당) — label은 쓰되 confidence 기반 설계는 피하세요.


In [ ]:
from agent.decisions import DecisionResult, _fallback

class LayaDecision:
    name = "laya"
    def __init__(self, router):
        self.router = router
    def decide(self, *, state_text, question, choices, decision_id):
        qs = {decision_id: {"type": "choice", "instructions": question,
                            "criteria": {c: None for c in choices}}}
        t0 = time.perf_counter()
        r = self.router.predict(state_text, qs, model="multilingual", max_len=8192)
        ms = (time.perf_counter() - t0) * 1000
        ans = r["answers"][decision_id]
        label, valid = ans["choice"], ans["choice"] in choices
        if not valid:
            label = _fallback(decision_id, choices)
        return DecisionResult(label=label, confidence=ans.get("confidence"),
                              latency_ms=round(ms, 2), valid=valid,
                              usage={}, cost_usd=0.0, raw=str(ans)[:200])

laya_dec = LayaDecision(router)
print("adapter ready")

In [ ]:
from agent.graph_v2 import build_graph_v2
from agent.decisions import SGLangDecision

client, llm = W.make_clients()
graph_laya   = build_graph_v2(laya_dec, llm)
graph_sglang = build_graph_v2(SGLangDecision(port=8001), llm)
sc = W.SCENARIOS[1]
print("시나리오:", sc["title"], "|", sc["question"])

In [ ]:
res_laya = W.run_verbose(graph_laya, sc["question"], sc["script"], "Laya")

In [ ]:
res_sglang = W.run_verbose(graph_sglang, sc["question"], sc["script"], "SGLang")

In [ ]:
W.compare(res_laya, res_sglang)

## 한국어 판단 품질 실측 (3 backend 비교)

이 워크샵의 한국어 160건(객관적 gold)으로 D1/D2를 전수 측정한 결과입니다 (the source project's decider-quality bench, the bench summary (source project)):

| backend | D1 guardrail (3지) | D2 routing strict/lenient (12지) | p50 latency |
|---|---|---|---|
| SGLang Qwen3-VL-8B | **0.913** | **0.629 / 0.707** | 33ms |
| strands-decider 2B | 0.613 | 0.500 / 0.593 | 85ms |
| Laya multilingual 322M | 0.419 | 0.357 / 0.379 | 21ms |

**해석**: 한국어 특화(Laya multilingual)보다 **모델 용량이 품질 상한을 지배**했습니다 (8B > 2B > 0.3B). Laya 측정치는 공식 문서의 zero-shot 수치(0.362)와 일치하며, Laya는 RLCD fine-tuning으로 0.766까지 오르는 **'튜닝 전제' 모델**입니다. 반대로 latency·풋프린트는 작을수록 유리합니다 — 품질/비용/속도의 3축 trade-off를 그대로 보여주는 결과입니다.


## 3. 정리 — decision model 선택지 지도

이 폴더(40_other_deciders)까지 포함하면, 같은 금융 상담 판단을 다섯 가지 방식으로 체험했습니다.

| 방식 | 판단 주체 | 판정 latency(이 환경) | 비용 구조 | 특징 |
|---|---|---|---|---|
| Bedrock Haiku 4.5 | 범용 LLM 생성 | ~640ms | 토큰 과금 | 품질 기준선, 관리형 |
| SGLang /v1/systemone (01-05) | 서빙 중인 Qwen3-VL-8B | **~35ms** | 한계비용 ~0 | 멀티모달, 기존 서빙 재사용 |
| Bedrock Intelligent Routing (06) | 판단이 아니라 **생성 모델 선택** | 선택된 모델의 생성 시간 | $1/1,000건 + 토큰 | 생성 비용 최적화 |
| strands-decider 2B (07) | 전용 decision model | ~80ms | self-host | calibrated confidence, Strands SDK 통합 |
| Laya 322-421M (08) | 전용 decision model | **~30ms** (in-process) | self-host (CPU 가능) | 다국어 자동 라우팅, 최소 풋프린트 |

- **한국어 관점**: Laya는 multilingual checkpoint가 자동 선택되어 한국어를 1급으로 다룹니다 — 이 워크샵의 한국어 시나리오와 궁합이 가장 좋은 전용 모델입니다.
- **주의**: confidence의 의미(캘리브레이션 방식)가 vendor마다 다릅니다. threshold는 제품 간 이식되지 않으므로, 반드시 자기 데이터로 reliability를 다시 측정하세요.
- zero-shot으로 부족하면 Laya는 RLCD fine-tuning 경로(공식 노트북)도 제공합니다 — typed-decisions benchmark에서 base 0.362 -> fine-tuned 0.766.